In [1]:
import os
import pandas as pd
import numpy as np

# Setup Path
base_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_path = os.path.join(base_path, 'data')
sub_path = os.path.join(base_path, 'submissions')

# Load Data dengan parse_dates untuk Timestamp
df_train = pd.read_csv(os.path.join(data_path, 'train.csv'), low_memory=False, parse_dates=['Timestamp'])
df_test = pd.read_csv(os.path.join(data_path, 'test.csv'), low_memory=False, parse_dates=['Timestamp'])

test_ids = df_test["Trip_ID"].copy()

print(f"Data Loaded. Train: {df_train.shape}, Test: {df_test.shape}")

Data Loaded. Train: (8000000, 25), Test: (4000000, 24)


In [9]:
def preprocess_data(df):
    # 1. JANGAN PAKAI dropna di data Test! 
    # Isi Timestamp yang kosong dengan nilai dummy agar tidak error saat .dt.hour
    df['Timestamp'] = df['Timestamp'].fillna(pd.Timestamp('2024-01-01 00:00:00'))
    
    # 2. Ekstrak fitur waktu (Jam)
    df['Hour'] = df['Timestamp'].dt.hour.astype(np.int8)
    
    # 3. Bersihkan Battery_Level
    df["Battery_Level"] = (
        df["Battery_Level"]
        .astype(str)
        .str.replace("%", "", regex=False)
    )
    
    # Konversi ke float, gunakan median untuk mengisi yang kosong
    df["Battery_Level"] = pd.to_numeric(df["Battery_Level"], errors='coerce')
    # Gunakan fillna dengan angka 0 atau median
    df["Battery_Level"] = df["Battery_Level"].fillna(0) 
    df["Battery_Level"] = df["Battery_Level"].astype(np.float32)
    
    # Drop Timestamp asli
    df = df.drop(columns=['Timestamp'], errors='ignore')
    return df

# Karena Cell 1 sudah jalan, kita harus Load ulang df_train & df_test SEBENTAR SAJA 
# agar data yang terlanjur "terhapus" tadi balik lagi.
df_train = pd.read_csv(os.path.join(data_path, 'train.csv'), low_memory=False, parse_dates=['Timestamp'])
df_test = pd.read_csv(os.path.join(data_path, 'test.csv'), low_memory=False, parse_dates=['Timestamp'])
test_ids = df_test["Trip_ID"].copy()

# Eksekusi fungsi yang sudah diperbaiki
df_train = preprocess_data(df_train)
df_test = preprocess_data(df_test)

# Sisa kode kategorikal (sama seperti sebelumnya)
cat_cols = ["Pickup_Zone", "Dropoff_Zone", "Car_Model", "Device_FP", 
            "Promo_Code", "Payment_Method", "Weather", "Traffic", "Signal_Strength"]

for col in cat_cols:
    df_train[col] = df_train[col].astype(str).replace(["nan", "None", "NaN"], "Unknown")
    df_test[col] = df_test[col].astype(str).replace(["nan", "None", "NaN"], "Unknown")

num_cols = df_train.select_dtypes(include=['float64', 'int64']).columns
df_train[num_cols] = df_train[num_cols].astype(np.float32)

print(f"Selesai! Train: {len(df_train)}, Test: {len(df_test)}") 
# Pastikan Test sekarang muncul 4000000

Selesai! Train: 8000000, Test: 4000000


In [10]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

# Gunakan 10% data dulu (800.000 baris) untuk RAM 16GB
df_sample = df_train.sample(frac=0.05, random_state=42)

X = df_sample.drop(columns=['Trip_ID', 'Trip_Label'])
y = df_sample['Trip_Label']

# CatBoost butuh list kolom kategorikal
x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=700, 
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1", 
    auto_class_weights='Balanced', 
    task_type="CPU",
    random_seed=42,
    verbose=100
)

model.fit(
    x_train, y_train,
    cat_features=cat_cols,
    eval_set=(x_val, y_val),
    early_stopping_rounds=50
)

0:	learn: 0.5287594	test: 0.5292082	best: 0.5292082 (0)	total: 1.71s	remaining: 19m 53s
100:	learn: 0.5763029	test: 0.5650273	best: 0.5650273 (100)	total: 2m 30s	remaining: 14m 49s
200:	learn: 0.6067896	test: 0.5727471	best: 0.5741330 (192)	total: 5m 37s	remaining: 13m 57s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5786104071
bestIteration = 234

Shrink model to first 235 iterations.


In [11]:
# 1. Pastikan fitur test diambil dari df_test yang asli agar jumlah barisnya tetap 4jt
feature_names = X.columns.tolist()

# Gunakan reindex untuk memastikan kolom sama persis dan urutan baris tidak berubah
x_test = df_test[feature_names]

# 2. Ambil prediksi
print(f"Sedang memprediksi {len(x_test)} baris...")
preds = model.predict(x_test)

# 3. Buat DataFrame
# Jika masih ada selisih, kita cek mana yang salah
if len(preds) != len(test_ids):
    print(f"WARNING: Jumlah prediksi ({len(preds)}) beda dengan ID ({len(test_ids)})")
    # Alternatif: Gunakan index dari x_test jika test_ids tidak sinkron
    submission = pd.DataFrame({
        "Trip_ID": test_ids[:len(preds)], # Potong agar sama (hanya untuk debug)
        "Trip_Label": preds.flatten()
    })
else:
    submission = pd.DataFrame({
        "Trip_ID": test_ids,
        "Trip_Label": preds.flatten()
    })

# 4. Save output
if not os.path.exists(sub_path):
    os.makedirs(sub_path)

output_file = os.path.join(sub_path, "submission_v1_frac0.05.csv")
submission.to_csv(output_file, index=False)
print(f"Selesai! File disimpan di: {output_file}")

Sedang memprediksi 4000000 baris...
Selesai! File disimpan di: /home/mmarufqk/DSMLComp/Ctrl+Alt+Del_DataVers_AnavaUGM/submissions/submission_v1_frac0.05.csv
